In [1]:
import cv2
import numpy as np
from tensorflow.keras.models import load_model

# ── Config ────────────────────────────────────────────────────────────────
VIDEO_PATH = "test.mp4"   # 🔴 change this
MODEL_PATH = "models/emotion_cnn.keras"
IMG_SIZE = 48

EMOTION_LABELS = [
    'angry', 'disgust', 'fear',
    'happy', 'sad', 'surprise', 'neutral'
]

# ── Load model (inference only) ───────────────────────────────────────────
model = load_model(MODEL_PATH, compile=False)

# ── Load face detector (SAFE) ─────────────────────────────────────────────
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

if face_cascade.empty():
    raise RuntimeError("❌ Haar cascade failed to load")

print("✅ Model & face detector loaded")

# ── Open video ────────────────────────────────────────────────────────────
cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    raise RuntimeError(f"❌ Cannot open video: {VIDEO_PATH}")

print("🎬 Processing video — press 'q' to quit")

# ── Main loop ─────────────────────────────────────────────────────────────
while True:
    ret, frame = cap.read()
    if not ret:
        break  # end of video

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.3,
        minNeighbors=5
    )

    for (x, y, w, h) in faces:
        face = gray[y:y+h, x:x+w]
        face = cv2.resize(face, (IMG_SIZE, IMG_SIZE))
        face = face.astype("float32") / 255.0
        face = np.expand_dims(face, axis=(0, -1))

        preds = model.predict(face, verbose=0)[0]
        emotion = EMOTION_LABELS[np.argmax(preds)]
        confidence = np.max(preds)

        # Draw box + label
        cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2)
        cv2.putText(
            frame,
            f"{emotion} ({confidence*100:.1f}%)",
            (x, y - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.9,
            (0, 255, 0),
            2
        )

    cv2.imshow("Emotion Detection (Video)", frame)

    if cv2.waitKey(25) & 0xFF == ord('q'):
        break

# ── Cleanup ───────────────────────────────────────────────────────────────
cap.release()
cv2.destroyAllWindows()
print("🛑 Video processing finished")

C:\Users\KIIT\AppData\Roaming\Python\Python313\site-packages\keras\src\saving\saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 78 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


✅ Model & face detector loaded
🎬 Processing video — press 'q' to quit
🛑 Video processing finished
